In [2]:
#!pip install langchain langchain-openai langchain-community faiss-cpu sentence-transformers langchain-text-splitters

In [3]:
import os

from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [4]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


In [5]:
llm = ChatOpenAI(
    model="gpt-4.1-nano",
    temperature=0
)

In [6]:
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)


/var/folders/9w/gbnr6ywd709g_25gy1np3x2h0000gn/T/ipykernel_8294/3617193625.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [7]:
raw_text = """
# Corrective Retrieval-Augmented Generation (C-RAG): Advanced System Notes

## 1. Retrieval Evaluator Architecture

C-RAG uses a fine-tuned T5-large model as a retrieval evaluator. The evaluator scores retrieved documents using a learned relevance-confidence objective. The output is a normalized confidence score in the range [0,1].

During training, the evaluator is optimized using supervised contrastive learning. Positive samples are documents that directly support the correct answer, while negative samples contain distractors or partially relevant information.

The evaluator does not directly modify document content. Instead, it influences downstream processing through threshold-based routing decisions.

---

## 2. Threshold Calibration

C-RAG uses two thresholds: τ_high and τ_low.

Thresholds are calibrated using a validation dataset. Calibration optimizes for a tradeoff between factual accuracy and retrieval coverage.

In practice, grid search is often used to select τ_high and τ_low. Some implementations apply Bayesian optimization for dynamic threshold tuning.

Domain-specific deployments may recalibrate thresholds to account for noise levels in domain corpora.

---

## 3. Decompose-Then-Recompose Refinement

The refinement process consists of:

1. Decomposition: splitting documents into semantic units ("knowledge strips").
2. Strip Scoring: assigning strip-level importance using inherited document confidence.
3. Recomposition: selecting high-confidence strips to reconstruct context.

Strip scoring may use normalized confidence scores derived from the evaluator.

Some implementations apply softmax weighting to convert strip confidence into selection probabilities.

Strips below a confidence threshold are discarded to reduce noise.

---

## 4. Ambiguous Retrieval Handling

If τ_low < score < τ_high, retrieval is ambiguous.

In ambiguous cases:

- Partial refinement is applied.
- External search is triggered.
- Internal strips are weighted higher than external strips to reduce contamination risk.

A weighted merge strategy is used to combine internal and external knowledge.

---

## 5. External Retrieval Module

External retrieval may include web search or API-based knowledge access.

External documents are embedded and scored similarly to internal documents.

To reduce misinformation risk, external strips are assigned lower default weights unless confirmed by internal evidence.

---

## 6. Failure Modes

C-RAG may still fail when:

- Evaluator confidence is overestimated.
- External sources contain coordinated misinformation.
- Thresholds are poorly calibrated.
- Strip scoring propagates document-level bias.

In low-resource languages, evaluator reliability may degrade.

---

## 7. Computational Tradeoffs

C-RAG introduces additional latency due to:

- Evaluator scoring
- Strip-level refinement
- Potential re-retrieval
- External search

However, this overhead often reduces hallucination rates and improves factual grounding.

---

## 8. Hallucination Reduction Mechanism

C-RAG reduces hallucination by:

- Discarding low-confidence documents
- Removing noisy strips
- Using threshold-based routing
- Enforcing context-constrained generation

Unlike traditional RAG, C-RAG does not rely solely on similarity search.

"""


In [8]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_text(raw_text)

docs = [Document(page_content=c) for c in chunks]



In [9]:

db = FAISS.from_documents(docs, embeddings)

retriever = db.as_retriever(search_kwargs={"k": 4})



In [10]:
# Answer Prompt
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer ONLY using the context."),
    ("human", """
Context:
{context}

Question:
{question}
""")
])


In [11]:
judge_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a strict technical reviewer. Be critical."),
    ("human", """
Question:
{question}

Answer:
{answer}

Evaluate this answer using these rules:

1. Does it fully address all parts of the question?
2. Does it explain mechanisms or reasoning, not just list terms?
3. Does it miss any important concepts implied by the question?
4. Does it rely on vague statements or say information is missing?

If ANY rule is violated, reply:

FAIL
MISSING: <what is missing or weak>
REWRITE_QUERY: <better query to retrieve missing info>

If all rules are satisfied, reply:

PASS
""")
])


In [12]:
query = "How do threshold calibration and strip-level weighting jointly prevent contamination from external sources in ambiguous CRAG retrieval cases"

In [13]:
docs_1 = retriever.invoke(query)

context_1 = "\n".join(d.page_content for d in docs_1)



In [14]:
# Initial Answer
msg_1 = answer_prompt.format_messages(
    context=context_1,
    question=query
)

answer_1 = llm.invoke(msg_1).content


print("\n--- Initial Answer ---")
print(answer_1)



--- Initial Answer ---
Threshold calibration and strip-level weighting jointly prevent contamination from external sources in ambiguous C-RAG retrieval cases by strategically managing the selection and weighting of retrieved strips. Calibration of τ_high and τ_low ensures that only strips with confidence scores within a certain range trigger ambiguous retrieval handling, thereby limiting the inclusion of low-confidence or potentially noisy external information. During ambiguous cases, internal strips are weighted more heavily than external strips, reducing the influence of external sources and minimizing contamination risk. Additionally, applying softmax weighting converts confidence scores into selection probabilities, further emphasizing higher-confidence internal strips. Discarding strips below a confidence threshold also helps eliminate noisy external information, ensuring that the retrieval process maintains factual accuracy and reduces contamination from external sources.


In [15]:
#  Judge
judge_msg = judge_prompt.format_messages(
    question=query,
    answer=answer_1
)

judge_output = llm.invoke(judge_msg).content


print("\n--- Judge Output ---")
print(judge_output)


--- Judge Output ---
FAIL  
MISSING: The explanation does not clearly specify how threshold calibration and strip-level weighting *jointly* work together in the process. It describes their individual roles but lacks detail on their interaction or combined effect in preventing external contamination. Additionally, it does not clarify how these mechanisms specifically address ambiguity in CRAG retrieval cases beyond general confidence management.  
REWRITE_QUERY: How do threshold calibration and strip-level weighting work together to prevent external contamination specifically in ambiguous CRAG retrieval scenarios?


In [16]:
# 4) Correction (If Needed)

better_query = None


if "FAIL" in judge_output:

    for line in judge_output.split("\n"):
        if line.startswith("REWRITE_QUERY:"):
            better_query = line.replace("REWRITE_QUERY:", "").strip()


In [17]:
if better_query:

    print("\n--- Better Query ---")
    print(better_query)

    docs_2 = retriever.invoke(better_query)

    context_2 = "\n".join(d.page_content for d in docs_2)

else:

    # No correction
    context_2 = context_1



--- Better Query ---
How do threshold calibration and strip-level weighting work together to prevent external contamination specifically in ambiguous CRAG retrieval scenarios?


In [18]:
#  Final Answer

final_msg = answer_prompt.format_messages(
    context=context_2,
    question=query
)

final_answer = llm.invoke(final_msg).content


print("\n=== Final Answer ---")
print(final_answer)



=== Final Answer ---
Threshold calibration and strip-level weighting jointly prevent contamination from external sources in ambiguous C-RAG retrieval cases by implementing a strategic decision-making process. Calibration of the thresholds τ_high and τ_low ensures that the system accurately identifies ambiguous retrievals, triggering partial refinement and external search only when necessary. During ambiguous cases, internal strips are weighted more heavily than external strips, reducing the influence of potentially contaminated external information. This approach allows the evaluator to influence downstream processing through threshold-based routing decisions without directly modifying document content, thereby minimizing contamination risk while maintaining retrieval accuracy.


In [19]:
print("\n--- Initial Answer ---")
print(answer_1)

print("\n--- Final Output ---")
print(final_answer)


--- Initial Answer ---
Threshold calibration and strip-level weighting jointly prevent contamination from external sources in ambiguous C-RAG retrieval cases by strategically managing the selection and weighting of retrieved strips. Calibration of τ_high and τ_low ensures that only strips with confidence scores within a certain range trigger ambiguous retrieval handling, thereby limiting the inclusion of low-confidence or potentially noisy external information. During ambiguous cases, internal strips are weighted more heavily than external strips, reducing the influence of external sources and minimizing contamination risk. Additionally, applying softmax weighting converts confidence scores into selection probabilities, further emphasizing higher-confidence internal strips. Discarding strips below a confidence threshold also helps eliminate noisy external information, ensuring that the retrieval process maintains factual accuracy and reduces contamination from external sources.

--- F

# The judge(LLM) said "fail" coz the repsonse is missing the right/correct info.
# And why did we get the second answer very similar to first is coz the knowledge lacks new info.